In [12]:
import os
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

import warnings

warnings.filterwarnings("ignore")

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [14]:
pkl_path = r'../../dataset/OralCancer_DatasetGabungan/dataset_paths.pkl'

with open(pkl_path, 'rb') as f:
    train_paths, train_labels, val_paths, val_labels, test_paths, test_labels = pickle.load(f)


EOFError: Ran out of input

In [3]:
class OralCancerDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


In [11]:
import pickle

save_path = r'../../dataset/OralCancer_DatasetGabungan/dataset_paths.pkl'

with open(save_path, 'wb') as f:
    pickle.dump((train_paths, train_labels, val_paths, val_labels, test_paths, test_labels), f)

print("✅ Dataset paths saved to:", save_path)


NameError: name 'train_paths' is not defined

In [9]:
import pickle

# GANTI dengan path ke file pickle yang benar
pkl_path = r'../../dataset/OralCancer_DatasetGabungan'

with open(pkl_path, 'rb') as f:
    train_paths, train_labels, val_paths, val_labels, test_paths, test_labels = pickle.load(f)


val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_dataset = OralCancerDataset(val_paths, val_labels, transform=val_test_transform)
test_dataset = OralCancerDataset(test_paths, test_labels, transform=val_test_transform)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


PermissionError: [Errno 13] Permission denied: 'C:\\Pytorch_project\\OralCancer_DatasetGabungan'

In [ ]:
def load_model(model_fn, checkpoint_path, classifier_layer):
    model = model_fn(pretrained=False)
    num_ftrs = classifier_layer(model)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(num_ftrs, 2)
    )
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.to(device)
    model.eval()
    return model

In [ ]:
mobilenet = load_model(models.mobilenet_v2, "../../models/final/MobilenetV2_best_model_20250716.pth", lambda m: m.classifier[1].in_features)
resnet = load_model(models.resnet50, "../../models/checkpoints/ResNet50_best_model_20250707.pth", lambda m: m.fc.in_features)
vgg = load_model(models.vgg16, "../../models/checkpoints/ResNet50_best_model_20250709.pth", lambda m: m.classifier[6].in_features)
densenet = load_model(models.densenet121, "../../models/checkpoints/ResNet50_best_model_20250714.pth", lambda m: m.classifier.in_features)


In [ ]:
def get_probs(models, dataloader):
    all_probs = [[] for _ in models]
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader):
            inputs = inputs.to(device)
            all_labels.extend(labels.numpy())

            for i, model in enumerate(models):
                outputs = model(inputs)
                probs = F.softmax(outputs, dim=1).cpu().numpy()
                all_probs[i].extend(probs)

    # Gabungkan seluruh model output: [n_samples, 8]
    stacked_probs = np.hstack(all_probs)
    return stacked_probs, np.array(all_labels)

In [ ]:
ensemble_models = [mobilenet, resnet, vgg, densenet]
X_val_stack, y_val_stack = get_probs(ensemble_models, val_loader)
X_test_stack, y_test_stack = get_probs(ensemble_models, test_loader)

In [ ]:
meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_val_stack, y_val_stack)

In [ ]:
y_pred_stack = meta_model.predict(X_test_stack)

In [ ]:
print("\n========== [ENSEMBLE STACKING RESULTS] ==========")
print(f"Test Accuracy: {accuracy_score(y_test_stack, y_pred_stack) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test_stack, y_pred_stack, target_names=['Non-Cancer', 'Cancer']))
